# VeloCity × Jev: Schadenmeldungen beurteilen

Kundinnen und Kunden von VeloCity Würzburg melden Schäden in freier Sprache. Jev, das System-One-Modell
von TypeSafe, beantwortet zu jeder Meldung fünf Fragen mit Wahrscheinlichkeiten. Welche Entscheidung
daraus folgt (sperren, prüfen, Wartungsauftrag, kein Schaden), legt der Code in `velocity_jev/regeln.py`
über Schwellen fest.

**Aufbau des Notebooks**

1. Einrichtung: Pakete, Projektordner, API-Key
2. Parameter: Fragen-Stand, Datensatz, Cache
3. Die fünf Fragen an Jev
4. Meldungen beurteilen
5. Kennzahlen
6. Alle Meldungen als interaktive Tabelle
7. Gruppierte Auswertung
8. Fehlentscheidungen und Kreuztabellen
9. Schwellen durchspielen
10. Fehleranalyse des Ausgangsstands
11. Vergleich der Fragen-Stände
12. Eine einzelne Meldung live beurteilen
13. Ergebnisse nach Supabase schreiben und in der Warenwirtschaft zeigen

Alle Antworten von Jev liegen im Cache `daten/cache/jev_cache.jsonl`. Ohne API-Key lässt sich das
Notebook vollständig ausführen, solange die Fragen nicht umformuliert werden.

## 1 Einrichtung

**API-Key in Deepnote hinterlegen** (nur nötig, wenn neue Anfragen an Jev gehen sollen):

1. Rechte Seitenleiste → **Integrations** → **+ Add integration** → **Environment variables**.
2. Name `TYPESAFE_API_KEY`, als Wert den Key aus der TypeSafe-Konsole eintragen, speichern.
3. Die Integration mit diesem Projekt verbinden (**Connect**) und die Maschine neu starten.

Deepnote legt den Wert verschlüsselt ab und stellt ihn als Umgebungsvariable bereit. Der Key gehört
nie in eine Notebook-Zelle, weil Zellen beim Teilen des Projekts sichtbar sind.

**Verbindung zur Datenbank** (nur für Abschnitt 13, nur im Projekt der Lehrperson): in derselben
Integration eine zweite Variable `DATABASE_URL` anlegen. Der Wert steht in `jev/.env` und verbindet sich
als Rolle `jev_schreiber`, die nur in das Schema `jev_labor` einfügen darf.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path


def projektordner_finden() -> Path:
    """Sucht den Ordner mit dem Paket velocity_jev: Arbeitsverzeichnis, ein Unterordner oder eine ZIP-Datei."""
    start = Path.cwd()
    for ordner in [start, *[p for p in start.iterdir() if p.is_dir()]]:
        if (ordner / "velocity_jev").is_dir():
            return ordner
    for archiv in sorted(start.glob("*.zip")):
        with zipfile.ZipFile(archiv) as z:
            if any(name.startswith("velocity_jev/") for name in z.namelist()):
                z.extractall(start)
                print(f"{archiv.name} entpackt.")
                return start
    raise FileNotFoundError("Ordner velocity_jev nicht gefunden. Bitte die Projektdateien hochladen.")


PROJEKT = projektordner_finden()
sys.path.insert(0, str(PROJEKT))
print("Projektordner:", PROJEKT)
print("Python:", sys.version.split()[0])
if sys.version_info < (3, 10):
    print("typesafe-sdk braucht Python 3.10 oder neuer: in Deepnote unter Environment eine neuere Version wählen.")

try:
    import typesafe_sdk  # noqa: F401
except ImportError:
    print("Installiere Pakete aus requirements.txt ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJEKT / "requirements.txt")], check=True)

In [ ]:
import pandas as pd
from IPython.display import display
from typesafe_sdk import TypeSafeClient

from velocity_jev.ablauf import klassifizieren
from velocity_jev.auswertung import (
    KOSTEN, STUFEN, alle_kennzahlen, gruppiert, kennzahlen_tabelle, schwellen_durchspielen, vergleich_bauen,
)
from velocity_jev.fragen import AKTUELLER_STAND, STAENDE, fragen
from velocity_jev.konfig import DATENSAETZE, ERGEBNISSE, MODELL, api_key_vorhanden
from velocity_jev.pipeline import Cache, beurteilen
from velocity_jev.protokoll import auswertung_ablegen, lauf_ablegen
from velocity_jev.regeln import PRUEFEN, entscheiden

pd.set_option("display.max_colwidth", 120)
print("Modell:", MODELL)
print("API-Key:", "gesetzt" if api_key_vorhanden() else "nicht gesetzt (nur Cache)")

## 2 Parameter

- `STAND`: Wortlaut der Fragen. Stand 0 ist die Übergabe, Stand 1 der beste Stand, Stand 2 wurde
  verworfen (siehe `docs/iterationen.md`).
- `DATENSATZ`: `"meldungen"` (48 Meldungen, an denen die Fragen verbessert wurden) oder `"holdout"`
  (18 Meldungen, die erst nach der Wahl des besten Stands gelaufen sind).
- `NUR_CACHE`: `True` liest nur gespeicherte Antworten und verbraucht keine Tokens. Bei `False` gehen
  Meldungen, die noch nicht im Cache liegen, an Jev; dafür ist der API-Key nötig.

In [ ]:
STAND = AKTUELLER_STAND
DATENSATZ = "meldungen"
NUR_CACHE = True

for nummer, beschreibung in STAENDE.items():
    print(f"Stand {nummer}: {beschreibung}{'   ← gewählt' if nummer == STAND else ''}")

## 3 Die fünf Fragen an Jev

Alle fünf Fragen gehen in einem Request an Jev und beziehen sich auf denselben State: den Meldungstext
und den Radtyp. Jev liest die Fragen wörtlich; was zur Erklärung eines Fehlers nötig wäre, gehört in
`instructions` oder `criteria`.

In [ ]:
def fragen_tabelle(stand: int) -> pd.DataFrame:
    """Die Fragen eines Stands mit Primitive, Anweisung und Kriterien als Tabelle."""
    zeilen = []
    for name, frage in fragen(stand).items():
        daten = frage.model_dump(mode="json")
        kriterien = daten.get("criteria")
        if isinstance(kriterien, dict):
            kriterien = " | ".join(f"{k}: {v}" for k, v in kriterien.items())
        elif isinstance(kriterien, list):
            kriterien = " | ".join(kriterien)
        zeilen.append({"frage": name, "primitive": type(frage).__name__,
                       "instructions": daten["instructions"], "criteria": kriterien or "–"})
    return pd.DataFrame(zeilen)


fragen_tabelle(STAND)

## 4 Meldungen beurteilen

Jede Meldung wird einmal an Jev geschickt oder aus dem Cache gelesen; danach wendet `regeln.py` die
Schwellen an. Jeder Lauf wird unter `ergebnisse/laeufe/<lauf_id>/` abgelegt und in
`ergebnisse/laeufe.csv` eingetragen.

In [ ]:
meldungen = pd.read_csv(DATENSAETZE[DATENSATZ], dtype={"meldung_id": str})

client = None
if not NUR_CACHE:
    if not api_key_vorhanden():
        raise RuntimeError("TYPESAFE_API_KEY fehlt, siehe Abschnitt 1. Mit NUR_CACHE = True geht es ohne Key.")
    client = TypeSafeClient(model=MODELL)

urteile, lauf = klassifizieren(meldungen, client=client, modell=MODELL, stand=STAND,
                               datensatz=DATENSATZ, anmerkung="Notebook", ausgabe=False)
if client is not None:
    client.close()
lauf_ablegen(ERGEBNISSE, lauf, urteile)

print(f"Lauf {lauf['lauf_id']}: {lauf['meldungen']} Meldungen beurteilt, {lauf['neue_requests']} neu an Jev")
print(f"Tokens dieses Laufs: {lauf['input_tokens']} Input, {lauf['output_tokens']} Output")
print(f"Tokens aller Urteile (auch aus dem Cache): {lauf['input_tokens_alle_urteile']} Input, "
      f"{lauf['output_tokens_alle_urteile']} Output")
if lauf["ohne_urteil"]:
    print("Ohne Urteil, weil nicht im Cache:", lauf["ohne_urteil"])

## 5 Kennzahlen

Die Soll-Entscheidung folgt aus den Labels in der CSV-Datei: kein Schaden → weiterleiten;
sicherheitsrelevant oder fahruntauglich → sperren; Schaden ohne passende Kategorie → prüfen;
sonst → Wartungsauftrag. Die Kosten beruhen auf groben Annahmen, die in `velocity_jev/auswertung.py`
stehen.

In [ ]:
vergleich = vergleich_bauen(meldungen, urteile)
kennzahlen = alle_kennzahlen(vergleich)
auswertung_ablegen(ERGEBNISSE, lauf, vergleich, kennzahlen)

print("Kostenannahmen je Fall:", KOSTEN)
kennzahlen_tabelle(kennzahlen).round(3)

## 6 Alle Meldungen

Eine Zeile je Meldung mit Soll und Ist. In Deepnote lässt sich die Tabelle über die Spaltenköpfe
sortieren und filtern; unter **Cell formatting** kann man etwa `sicherheitsrelevant` als Farbskala
einfärben. Die Spalte `ergebnis` fasst zusammen, ob die Entscheidung richtig war und welche Art von
Fehler vorliegt.

In [ ]:
SPALTEN = [
    "meldung_id", "merkmal", "typ_code", "text", "ergebnis", "soll_entscheidung", "entscheidung", "begruendung",
    "soll_kategorie", "kategorie", "kategorie_konfidenz", "soll_schwere", "schwere_stufe", "schwere_score",
    "schwere_konfidenz", "soll_ist_schaden", "ist_schadensmeldung", "soll_sicherheitsrelevant",
    "sicherheitsrelevant", "soll_personenschaden", "personenschaden", "eskalation",
]
tabelle = vergleich[SPALTEN].round(2)
tabelle

## 7 Gruppierte Auswertung

Die Deepnote-Tabelle sortiert und filtert, gruppiert aber nicht. Die Gruppierung übernimmt deshalb
pandas. `GRUPPIEREN_NACH` lässt sich auf jede Spalte der Tabelle aus Abschnitt 6 setzen, sinnvoll sind
etwa `"merkmal"`, `"ergebnis"`, `"soll_kategorie"`, `"typ_code"` oder `"soll_entscheidung"`.
Ohne Code geht es in Deepnote auch mit einem **Pivot table**-Block auf die Variable `tabelle`.

In [ ]:
GRUPPIEREN_NACH = "merkmal"
gruppiert(vergleich, GRUPPIEREN_NACH)

## 8 Fehlentscheidungen und Kreuztabellen

Zuerst alle Meldungen, bei denen der Code anders entschieden hat als das Soll, danach die
Kreuztabellen der Entscheidung und der Schwere (Zeilen: Soll, Spalten: Ergebnis).

In [ ]:
abweichungen = tabelle[tabelle.entscheidung != tabelle.soll_entscheidung]
print(f"{len(abweichungen)} von {len(tabelle)} Entscheidungen weichen vom Soll ab.")
abweichungen[["meldung_id", "merkmal", "ergebnis", "soll_entscheidung", "entscheidung", "begruendung",
              "sicherheitsrelevant", "ist_schadensmeldung", "schwere_stufe", "text"]]

In [ ]:
entscheidungen = pd.crosstab(vergleich.soll_entscheidung, vergleich.entscheidung,
                             rownames=["Soll"], colnames=["Code"])
schaden = vergleich[vergleich.soll_ist_schaden]
reihenfolge = list(STUFEN)
schwere = (pd.crosstab(schaden.soll_schwere, schaden.schwere_stufe, rownames=["Soll"], colnames=["Jev"])
           .reindex(index=reihenfolge, columns=reihenfolge, fill_value=0))
display(entscheidungen)
display(schwere)

## 9 Schwellen durchspielen

Die Wahrscheinlichkeiten ändern sich nicht, wenn man Schwellen verschiebt; deshalb kostet diese
Rechnung keine Tokens. Die Tabelle ist nach Kosten sortiert. Eine Schwelle, die nur auf diesen
Meldungen optimal ist, gilt nicht automatisch für neue Meldungen.

In [ ]:
schwellen_durchspielen(vergleich).round(2)

## 10 Fehleranalyse des Ausgangsstands

Für Stand 0 wurde jede Fehlentscheidung einer von vier Ursachen zugeordnet: Frage unklar, Label
strittig, bekannte Grenze von Jev, Fehler im Code. Die Zuordnung ist eine Einschätzung, keine Messung.

In [ ]:
fehleranalyse = pd.read_csv(PROJEKT / "docs" / "fehleranalyse_stand0.csv", dtype={"meldung_id": str})
display(pd.crosstab(fehleranalyse.merkmal, fehleranalyse.ursache, margins=True, margins_name="Summe"))
fehleranalyse

## 11 Vergleich der Fragen-Stände

Jeder Stand und jeder Datensatz wird aus dem Cache ausgewertet; es entstehen keine Kosten.
Die Kennzahlen stehen je Lauf nebeneinander.

In [ ]:
def stand_auswerten(stand: int, datensatz: str) -> dict:
    """Einen Fragen-Stand auf einem Datensatz nur aus dem Cache auswerten."""
    daten = pd.read_csv(DATENSAETZE[datensatz], dtype={"meldung_id": str})
    u, l = klassifizieren(daten, client=None, modell=MODELL, stand=stand, datensatz=datensatz, ausgabe=False)
    if len(u) < len(daten):
        return {"datensatz": datensatz, "stand": stand, "hinweis": f"{len(daten) - len(u)} Meldungen nicht im Cache"}
    k = alle_kennzahlen(vergleich_bauen(daten, u))
    return {"datensatz": datensatz, "stand": stand, "beschreibung": STAENDE[stand],
            "tokens_input": l["input_tokens_alle_urteile"], **k}


vergleich_staende = pd.DataFrame([stand_auswerten(s, d) for d in DATENSAETZE for s in STAENDE])
vergleich_staende.round(3)

## 12 Eine einzelne Meldung live beurteilen

Braucht den API-Key. Ein neuer Text kostet einen Request mit etwa 1 400 Input-Tokens; derselbe Text
kommt beim zweiten Mal aus dem Cache.

In [ ]:
TEXT = "Bremse hinten quietscht und greift erst ganz spät"
RADTYP = "CITY"   # CITY, EBIKE oder CARGO

if api_key_vorhanden():
    with TypeSafeClient(model=MODELL) as c:
        u = beurteilen("LIVE", TEXT, RADTYP, client=c, modell=MODELL, cache=Cache(), stand=STAND)
    e = entscheiden(u)
    display(pd.DataFrame([
        {"Urteil": "ist_schadensmeldung", "Wert": u.ist_schadensmeldung},
        {"Urteil": "sicherheitsrelevant", "Wert": u.sicherheitsrelevant},
        {"Urteil": "personenschaden", "Wert": u.personenschaden},
        {"Urteil": f"Kategorie {u.kategorie} (Konfidenz)", "Wert": u.kategorie_konfidenz},
        {"Urteil": f"Schwere {u.schwere_stufe} (Erwartungswert)", "Wert": u.schwere_score},
    ]).round(2))
    print(f"Entscheidung: {e.entscheidung}{' + Eskalation an den Kundendienst' if e.eskalation else ''}")
    print(f"Begründung: {e.begruendung}")
    print("Aus dem Cache." if u.aus_cache else f"Neuer Request: {u.input_tokens} Input-, {u.output_tokens} Output-Tokens.")
else:
    print("Kein API-Key gesetzt, siehe Abschnitt 1.")

## 13 Ergebnisse nach Supabase schreiben

Der Lauf aus Abschnitt 4 wird mit der Rolle `jev_schreiber` in das Schema `jev_labor` geschrieben:
Meldungen, Lauf und Urteile in einer Transaktion. Ein Lauf, der schon in der Datenbank steht, wird
nicht doppelt geschrieben. In der Warenwirtschaft (Instandhaltung, Reiter „Meldungseingang“) erscheint
nur ein **freigegebener** Lauf, und zwar der zuletzt freigegebene. Dort übernimmt oder verwirft die
Werkstatt jeden Vorschlag; erst die Übernahme legt eine Schadensmeldung an.

Beide Schalter stehen auf `False`. Eine Freigabe ändert, was die Werkstatt sieht, und bleibt deshalb
eine bewusste Entscheidung.

In [ ]:
SCHREIBEN = False   # True: den Lauf aus Abschnitt 4 nach Supabase schreiben
FREIGEBEN = False   # True: den Lauf zusätzlich in der Warenwirtschaft zeigen

if SCHREIBEN:
    url = os.environ.get("DATABASE_URL", "")
    if not url or "USER:PASSWORT" in url:
        raise RuntimeError("DATABASE_URL fehlt, siehe Abschnitt 1.")
    from velocity_jev.datenbank import lauf_hochladen
    ergebnis = lauf_hochladen(url, lauf, urteile, meldungen, freigeben=FREIGEBEN)
    stand = "neu geschrieben" if ergebnis["neu"] else "war schon vorhanden"
    print(f"Lauf {ergebnis['lauf_id']} {stand}, {ergebnis['urteile']} Urteile"
          f"{', freigegeben' if ergebnis['freigegeben'] else ''}.")
else:
    print("Nichts geschrieben. SCHREIBEN = True setzen, um den Lauf nach Supabase zu übertragen.")